專案名稱：基於 YOLO 之宿舍環境智慧垃圾辨識系統
開發環境： Google Colab (雲端訓練) / Anaconda (本地部署)
開發語言： Python
核心技術： YOLO 影像辨識、Label Studio 資料標註、OpenCV 攝像頭串接

In [ ]:
!nvidia-smi

In [ ]:
import os
from google.colab import drive

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# 2. 複製資料 (確保路徑正確)
!cp /content/drive/MyDrive/YOLO_Project/data.zip /content/

# 3. 整合：解壓縮並同時進行檢查
# 加上 -o 參數代表「強制覆蓋」，加上 -q 代表「安靜執行 (不列印一堆檔案名)」
!unzip -o -q /content/data.zip -d /content/custom_data

if os.path.exists('/content/custom_data'):
    print("✅ 解壓縮成功！已自動覆蓋舊檔案並建立於 /content/custom_data")
    # 檢查內容是否包含 images 和 labels
    items = os.listdir('/content/custom_data')
    print("資料夾內容：", items[:5])
else:
    print("❌ 錯誤：解壓縮失敗，請檢查 /content/data.zip 是否存在。")

In [ ]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

In [ ]:
!pip install ultralytics

In [ ]:
import yaml

# 根據你的截圖，資料夾名稱是 'data' 而不是 'custom_data'
classes_path = '/content/custom_data/classes.txt' # 假設 classes.txt 還是在這裡

with open(classes_path, 'r') as f:
    classes = [line.strip() for line in f.readlines() if line.strip()]

data = {
    'path': '/content/data',        # 修正為截圖中的資料夾名稱
    'train': 'train/images',
    'val': 'validation/images',
    'nc': len(classes),
    'names': classes
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data, f, sort_keys=False)

print("✅ YAML 路徑已對齊！")
!cat /content/data.yaml

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640

In [ ]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=data/validation/images save=True

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')

In [ ]:
# Create "my_model" folder to store model weights and train results
!mkdir /content/my_model
!cp /content/runs/detect/train/weights/best.pt /content/my_model/my_model.pt
!cp -r /content/runs/detect/train /content/my_model

# Zip into "my_model.zip"
%cd my_model
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip train
%cd /content

In [ ]:
import os

# 1. 確保雲端硬碟已掛載
from google.colab import drive
drive.mount('/content/drive')

# 2. 定義備份路徑
backup_path = '/content/drive/MyDrive/YOLO_Project/Core_Backup'
!mkdir -p {backup_path}

print("🚀 正在搬運核心重點到雲端...")

# 3. 搬運訓練成果 (包含 best.pt 和 results.png)
if os.path.exists('/content/runs/detect/train'):
    !cp -r /content/runs/detect/train {backup_path}/
    print("✅ 訓練成果 (runs) 已備份")

# 4. 搬運設定檔與處理腳本
if os.path.exists('/content/data.yaml'):
    !cp /content/data.yaml {backup_path}/
    print("✅ data.yaml 已備份")

if os.path.exists('/content/train_val_split.py'):
    !cp /content/train_val_split.py {backup_path}/
    print("✅ train_val_split.py 已備份")

print(f"\n🎉 重點備份完成！地點：{backup_path}")

In [ ]:
# 1. 掛載硬碟
from google.colab import drive
drive.mount('/content/drive')

# 2. 定義雲端備份的位置
backup_path = '/content/drive/MyDrive/YOLO_Project/Core_Backup'

print("📥 正在從雲端還原重點檔案...")

# 3. 把東西搬回 Colab 的臨時空間 (/content)
!cp -r {backup_path}/train /content/runs/detect/ 2>/dev/null
!cp {backup_path}/data.yaml /content/ 2>/dev/null
!cp {backup_path}/train_val_split.py /content/ 2>/dev/null

print("✅ 還原完成！你現在可以用上次訓練好的 best.pt 了。")
!ls -F /content/

In [ ]:
# 3. 把東西搬回 Colab 的臨時空間 (/content)
# 先建立目標路徑，確保 cp 有地方可以放
!mkdir -p /content/runs/detect/

!cp -r {backup_path}/train /content/runs/detect/
!cp {backup_path}/data.yaml /content/
!cp {backup_path}/train_val_split.py /content/

1. 業務流程 (Business Process / Workflow)系統架構是「骨架」，而業務流程就是「血液」。它描述的是人在使用系統時的步驟。在你的專案中：流程是「拍攝照片 → 進入 Label Studio 標註 → 匯出至 Colab 訓練 → 得到權重後下載至本地端 → 開啟攝影機進行辨識」。書審價值：展示你了解如何將技術轉化為實際的操作程序。2. 資料庫設計 (Database Schema / Data Model)資管系非常看重資料如何被組織與儲存。在你的專案中：雖然你目前沒有使用複雜的資料庫，但你的 data.zip 結構（images 與 labels 的對應關係）以及 data.yaml 的類別定義，就是一種資料模型 (Data Model)。書審價值：展示你具備資料標準化與組織的能力。3. 使用者介面與體驗 (UI/UX Design)這關係到系統好不好用。在你的專案中：你在執行 yolo_detect.py 時，畫面呈現出的邊框 (BBox)、類別標籤與信心值 (Confidence Score)，就是最直接的 User Interface。書審價值：說明你如何優化視覺回饋，讓使用者能直觀地理解 AI 辨識的結果。4. 系統測試與驗證 (System Testing & Validation)資管強調「品質管理」，程式能跑不代表它是完美的。在你的專案中：你提到「辨識度不太高」，這其實就是測試後的 驗證結果 (Validation)。書審價值：放上 confusion_matrix.png (混淆矩陣) 或 results.png，並分析為什麼某些物件會誤判，這展現了你具備專業的除錯與分析能力。總結：資管系看專案的四個層次層次核心問題你的實踐例子架構層 (Architecture)系統長什麼樣子？雲端訓練、本地部署、自動備份腳本。流程層 (Process)資料怎麼跑？從圖片標註到即時影像辨識的自動化管線。資料層 (Data)內容存哪裡？YAML 設定檔、Images/Labels 資料結構。評估層 (Evaluation)效果好不好？訓練出的損失曲線 (Loss Curve) 與實際辨識準確率。

1. 需求分析與問題解決 (Problem Solving & Logic)這部分展現你的「資管思維」，說明你如何觀察問題並提出解決方案。開發動機：為什麼選擇 YOLO？（例如：追求實時辨識的效率與輕量化）。遇到的挑戰與對策：挑戰：雲端環境（Colab）具備揮發性，斷線後訓練數據會遺失。對策：自行撰寫自動化備份腳本，將核心資產（Core Backup）同步至 Google Drive。挑戰：資料解壓縮時的互動式衝突（重複詢問 y/n）。對策：導入靜默覆蓋參數 (-o -q)，實現無人值守的自動化流程。2. 測試與評估指標 (Testing & Evaluation)程式能跑只是基本，資管系更在乎你如何證明系統「好不好」。效能評估 (Metrics)：Loss 曲線分析：放入 results.png，解釋損失函數下降的趨勢（證明模型有在學習）。混淆矩陣 (Confusion Matrix)：展示 confusion_matrix.png，分析哪些類別辨識度高，哪些容易誤判。實時部署測試：說明在本地端使用 CUDA 加速後的 FPS 表現，以及在 720p 解析度下的流暢度。3. 未來優化與擴充性 (Future Roadmap)這頁最能體現你的成長潛力，即便現在模型不完美（如你提到的辨識度不高），也要有專業的優化計畫。資料增強 (Data Augmentation)：計畫蒐集更多不同光影、角度的照片，以提升模型的魯棒性（Robustness）。超參數微調 (Hyperparameter Tuning)：未來將嘗試調整 Learning Rate 或增加 Epochs，尋找最佳收斂點。跨平台整合：構思如何將辨識結果串接至網頁端（FastAPI）或資料庫（SQLite），實現完整的資訊管理系統。💡 總結 PPT 頁面配置建議頁碼標題重點內容P1專案概述題目、技術棧、YouTube 成果演示 QR Code。P2系統架構圖雲端訓練、自動備份與本地部署的資料流。P3開發邏輯與難點自動化腳本開發、災難復原機制（備份至雲端）。P4實驗數據分析results.png、confusion_matrix.png 截圖與分析。P5結語與展望誠實分析現狀（辨識度問題）與具體優化步驟。